In [ ]:
from __future__ import annotations

from inspect import signature
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / '.git').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.experiments.registry import FEATURE_REGISTRY
from src.features import (
    add_adx_features,
    add_atr_features,
    add_bollinger_features,
    add_close_returns,
    add_feature_transforms,
    add_lagged_features,
    add_macd_features,
    add_macro_context_features,
    add_mfi_features,
    add_ppo_features,
    add_price_momentum_features,
    add_regime_context_features,
    add_return_momentum_features,
    add_roc_features,
    add_rsi_features,
    add_session_context_features,
    add_shock_context_features,
    add_stochastic_features,
    add_support_resistance_features,
    add_support_resistance_v2_features,
    add_trend_features,
    add_trend_regime_features,
    add_vol_normalized_momentum_features,
    add_volatility_features,
    add_volume_features,
)
from src.plots import (
    plot_price_indicator_panel,
    plot_price_overlay,
    plot_price_with_feature_combo,
    plot_price_with_features,
    plotly_chart_config,
)
from src.src_data import load_ohlcv_csv

pd.options.display.float_format = '{:,.6f}'.format


def show_fig(fig):
    fig.show(config=plotly_chart_config())


def existing_columns(frame: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in frame.columns]


def apply_feature(frame: pd.DataFrame, feature_fn, **params):
    before_cols = list(frame.columns)
    out = feature_fn(frame.copy(), **params)
    new_cols = [column for column in out.columns if column not in before_cols]
    return out, new_cols


def show_feature_table(frame: pd.DataFrame, columns: list[str], rows: int = 5):
    if not columns:
        print('No new columns were produced.')
        return
    display(frame.loc[:, columns].tail(rows))


In [ ]:
FEATURE_DESCRIPTIONS = {
    'returns': 'Simple or log returns from the selected price series.',
    'volatility': 'Rolling and EWMA volatility from an existing returns column.',
    'trend': 'SMA and EMA overlays plus price-over-average ratios.',
    'trend_regime': 'Trend-state context from short and long moving-average relationships.',
    'lags': 'Lagged copies of selected columns for causal comparisons.',
    'bollinger': 'Bollinger moving average, upper/lower bands, width, and percent-b.',
    'macd': 'MACD line, signal line, and histogram from EMA spreads.',
    'ppo': 'Percentage Price Oscillator, signal line, and histogram.',
    'roc': 'Rate-of-change over one or more lookback windows.',
    'atr': 'Average True Range and optional ATR-over-price normalization.',
    'adx': 'Average Directional Index trend-strength measure.',
    'volume_features': 'Volume z-score and volume-vs-range context features.',
    'mfi': 'Money Flow Index using price and volume together.',
    'rsi': 'Relative Strength Index for one or more windows.',
    'stochastic': 'Stochastic %K and %D oscillator from close vs high-low range.',
    'price_momentum': 'Price-based momentum over several horizons.',
    'return_momentum': 'Momentum from a returns column over several horizons.',
    'vol_normalized_momentum': 'Momentum normalized by recent volatility.',
    'session_context': 'Hour, weekday, and session membership context features.',
    'regime_context': 'Volatility/trend regime labels from short-vs-long context.',
    'shock_context': 'Shock/reversion context from returns, ATR, and distance from mean.',
    'support_resistance': 'Rolling support and resistance levels plus distance features.',
    'support_resistance_v2': 'Pivot-based support/resistance context with touch and breakout logic.',
    'macro_context': 'Lagged and transformed exogenous or macro columns.',
    'feature_transforms': 'Post-feature transforms such as rolling clip, ratio, and rolling z-score.',
}

feature_reference = pd.DataFrame(
    [
        {
            'feature': name,
            'function': fn.__name__,
            'signature': str(signature(fn)),
            'what_it_computes': FEATURE_DESCRIPTIONS.get(name, ''),
        }
        for name, fn in sorted(FEATURE_REGISTRY.items())
    ]
)

display(
    Markdown(
        '## Available Features\n\n'
        + '\n'.join(f'- `{row.feature}`: {row.what_it_computes}' for row in feature_reference.itertuples(index=False))
    )
)

feature_reference


In [ ]:
DATA_PATH = repo_root / 'data/raw/dukas_copy_bank/usdjpy_h1.csv'
DATA_SYMBOL = 'USDJPY'
START = '2017-05-07 23:00:00'
END = '2024-12-31 14:00:00'

raw_frame = load_ohlcv_csv(DATA_PATH, symbol=DATA_SYMBOL, start=START, end=END)
raw_frame.head()


In [ ]:
RETURNS_LOG = True
RETURNS_COL_NAME = None

returns_frame, returns_new_cols = apply_feature(
    raw_frame,
    add_close_returns,
    log=RETURNS_LOG,
    col_name=RETURNS_COL_NAME,
)
show_feature_table(returns_frame, returns_new_cols)
show_fig(
    plot_price_with_features(
        returns_frame,
        title=f'Close vs Returns (log={RETURNS_LOG})',
        feature_cols=returns_new_cols,
        normalize=False,
    )
)


In [ ]:
VOL_RETURNS_COL = 'close_logret'
VOL_ROLLING_WINDOWS = [10, 20, 60]
VOL_EWMA_SPANS = [10, 20]
VOL_ANNUALIZATION_FACTOR = None

vol_base = add_close_returns(raw_frame, log=True)
vol_base = add_close_returns(vol_base, log=False)
vol_frame, vol_new_cols = apply_feature(
    vol_base,
    add_volatility_features,
    returns_col=VOL_RETURNS_COL,
    rolling_windows=VOL_ROLLING_WINDOWS,
    ewma_spans=VOL_EWMA_SPANS,
    annualization_factor=VOL_ANNUALIZATION_FACTOR,
)
show_feature_table(vol_frame, vol_new_cols)
show_fig(
    plot_price_with_features(
        vol_frame,
        title=f'Close vs Volatility from {VOL_RETURNS_COL}',
        feature_cols=vol_new_cols,
        normalize=False,
    )
)


In [ ]:
TREND_SMA_WINDOWS = []
TREND_EMA_SPANS = [9, 21, 50]

trend_frame, trend_new_cols = apply_feature(
    raw_frame,
    add_trend_features,
    price_col='close',
    sma_windows=TREND_SMA_WINDOWS,
    ema_spans=TREND_EMA_SPANS,
)
trend_overlay_cols = ['close'] + [f'close_sma_{window}' for window in TREND_SMA_WINDOWS] + [f'close_ema_{span}' for span in TREND_EMA_SPANS]
trend_ratio_cols = [column for column in trend_new_cols if '_over_' in column]
show_feature_table(trend_frame, trend_new_cols)
show_fig(plot_price_overlay(trend_frame, existing_columns(trend_frame, trend_overlay_cols), title=f'Close with SMA {TREND_SMA_WINDOWS} and EMA {TREND_EMA_SPANS}'))
show_fig(
    plot_price_with_features(
        trend_frame,
        title='Price over SMA/EMA ratios',
        feature_cols=trend_ratio_cols,
        normalize=False,
    )
)


In [ ]:
TREND_REGIME_BASE_SMA = 50
TREND_REGIME_SHORT_SMA = 20
TREND_REGIME_LONG_SMA = 50

trend_regime_sma_windows = sorted({TREND_REGIME_BASE_SMA, TREND_REGIME_SHORT_SMA, TREND_REGIME_LONG_SMA})
trend_regime_base = add_trend_features(raw_frame, price_col='close', sma_windows=trend_regime_sma_windows, ema_spans=[])
trend_regime_frame, trend_regime_new_cols = apply_feature(
    trend_regime_base,
    add_trend_regime_features,
    price_col='close',
    base_sma_for_sign=TREND_REGIME_BASE_SMA,
    short_sma=TREND_REGIME_SHORT_SMA,
    long_sma=TREND_REGIME_LONG_SMA,
)
trend_regime_overlay_cols = ['close'] + [f'close_sma_{window}' for window in trend_regime_sma_windows]
show_feature_table(trend_regime_frame, trend_regime_new_cols)
show_fig(plot_price_overlay(trend_regime_frame, existing_columns(trend_regime_frame, trend_regime_overlay_cols), title='Close with trend-regime SMAs'))
show_fig(
    plot_price_with_features(
        trend_regime_frame,
        title='Trend regime states',
        feature_cols=trend_regime_new_cols,
        normalize=False,
    )
)


In [ ]:
LAG_COLUMNS = ['close', 'volume']
LAG_PERIODS = [1, 2, 5]
LAG_PREFIX = 'lag'

lag_frame, lag_new_cols = apply_feature(
    raw_frame,
    add_lagged_features,
    cols=LAG_COLUMNS,
    lags=LAG_PERIODS,
    prefix=LAG_PREFIX,
)
lag_overlay_cols = ['close'] + [f'{LAG_PREFIX}_close_{lag}' for lag in LAG_PERIODS if 'close' in LAG_COLUMNS]
show_feature_table(lag_frame, lag_new_cols)
if len(existing_columns(lag_frame, lag_overlay_cols)) > 1:
    show_fig(plot_price_overlay(lag_frame, existing_columns(lag_frame, lag_overlay_cols), title='Close with lagged close columns'))
show_fig(
    plot_price_with_features(
        lag_frame,
        title=f'Lagged columns: {LAG_COLUMNS}',
        feature_cols=lag_new_cols,
        normalize=False,
    )
)


In [ ]:
BB_WINDOW = 20
BB_N_STD = 2.0

bb_frame, bb_new_cols = apply_feature(
    raw_frame,
    add_bollinger_features,
    price_col='close',
    window=BB_WINDOW,
    n_std=BB_N_STD,
)
bb_overlay_cols = ['close', f'bb_ma_{BB_WINDOW}', f'bb_upper_{BB_WINDOW}_{BB_N_STD}', f'bb_lower_{BB_WINDOW}_{BB_N_STD}']
bb_feature_cols = [column for column in bb_new_cols if column not in {f'bb_ma_{BB_WINDOW}', f'bb_upper_{BB_WINDOW}_{BB_N_STD}', f'bb_lower_{BB_WINDOW}_{BB_N_STD}'}]
show_feature_table(bb_frame, bb_new_cols)
show_fig(plot_price_overlay(bb_frame, existing_columns(bb_frame, bb_overlay_cols), title=f'Close with Bollinger Bands ({BB_WINDOW}, {BB_N_STD})'))
show_fig(
    plot_price_with_features(
        bb_frame,
        title='Bollinger derived features',
        feature_cols=bb_feature_cols,
        normalize=False,
    )
)


In [ ]:
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9

macd_frame, macd_new_cols = apply_feature(
    raw_frame,
    add_macd_features,
    price_col='close',
    fast=MACD_FAST,
    slow=MACD_SLOW,
    signal=MACD_SIGNAL,
)
macd_line_col = f'macd_{MACD_FAST}_{MACD_SLOW}'
macd_signal_col = f'macd_signal_{MACD_SIGNAL}'
macd_hist_col = f'macd_hist_{MACD_FAST}_{MACD_SLOW}_{MACD_SIGNAL}'
show_feature_table(macd_frame, macd_new_cols)
show_fig(
    plot_price_indicator_panel(
        macd_frame,
        price_cols=['close'],
        line_cols=[macd_line_col, macd_signal_col],
        bar_cols=[macd_hist_col],
        title=f'Close vs MACD ({MACD_FAST}, {MACD_SLOW}, {MACD_SIGNAL})',
    )
)


In [ ]:
PPO_FAST = 12
PPO_SLOW = 26
PPO_SIGNAL = 9

ppo_frame, ppo_new_cols = apply_feature(
    raw_frame,
    add_ppo_features,
    price_col='close',
    fast=PPO_FAST,
    slow=PPO_SLOW,
    signal=PPO_SIGNAL,
)
ppo_line_col = f'ppo_{PPO_FAST}_{PPO_SLOW}'
ppo_signal_col = f'ppo_signal_{PPO_SIGNAL}'
ppo_hist_col = f'ppo_hist_{PPO_FAST}_{PPO_SLOW}_{PPO_SIGNAL}'
show_feature_table(ppo_frame, ppo_new_cols)
show_fig(
    plot_price_indicator_panel(
        ppo_frame,
        price_cols=['close'],
        line_cols=[ppo_line_col, ppo_signal_col],
        bar_cols=[ppo_hist_col],
        title=f'Close vs PPO ({PPO_FAST}, {PPO_SLOW}, {PPO_SIGNAL})',
    )
)


In [ ]:
ROC_WINDOWS = [10, 20]

roc_frame, roc_new_cols = apply_feature(
    raw_frame,
    add_roc_features,
    price_col='close',
    windows=ROC_WINDOWS,
)
show_feature_table(roc_frame, roc_new_cols)
show_fig(
    plot_price_with_features(
        roc_frame,
        title=f'Close vs ROC {ROC_WINDOWS}',
        feature_cols=roc_new_cols,
        normalize=False,
    )
)


In [ ]:
ATR_WINDOW = 14
ATR_METHOD = 'wilder'
ATR_ADD_OVER_PRICE = True

atr_frame, atr_new_cols = apply_feature(
    raw_frame,
    add_atr_features,
    high_col='high',
    low_col='low',
    close_col='close',
    window=ATR_WINDOW,
    method=ATR_METHOD,
    add_over_price=ATR_ADD_OVER_PRICE,
)
show_feature_table(atr_frame, atr_new_cols)
show_fig(
    plot_price_indicator_panel(
        atr_frame,
        price_cols=['close'],
        line_cols=atr_new_cols,
        bar_cols=[],
        title=f'Close vs ATR ({ATR_WINDOW}, {ATR_METHOD})',
    )
)


In [ ]:
ADX_WINDOW = 9

adx_frame, adx_new_cols = apply_feature(
    raw_frame,
    add_adx_features,
    high_col='high',
    low_col='low',
    close_col='close',
    window=ADX_WINDOW,
)
show_feature_table(adx_frame, adx_new_cols)
show_fig(
    plot_price_indicator_panel(
        adx_frame,
        price_cols=['close'],
        line_cols=adx_new_cols,
        bar_cols=[],
        title=f'Close vs ADX ({ADX_WINDOW})',
    )
)


In [ ]:
VOLUME_FEATURES_ATR_WINDOW = 14
VOLUME_FEATURES_VOL_Z_WINDOW = 20

volume_feature_frame, volume_feature_new_cols = apply_feature(
    raw_frame,
    add_volume_features,
    volume_col='volume',
    high_col='high',
    low_col='low',
    close_col='close',
    atr_window=VOLUME_FEATURES_ATR_WINDOW,
    vol_z_window=VOLUME_FEATURES_VOL_Z_WINDOW,
)
show_feature_table(volume_feature_frame, volume_feature_new_cols)
show_fig(
    plot_price_with_features(
        volume_feature_frame,
        title='Close vs volume-derived features',
        feature_cols=volume_feature_new_cols,
        normalize=False,
    )
)


In [ ]:
MFI_WINDOW = 14

mfi_frame, mfi_new_cols = apply_feature(
    raw_frame,
    add_mfi_features,
    high_col='high',
    low_col='low',
    close_col='close',
    volume_col='volume',
    window=MFI_WINDOW,
)
show_feature_table(mfi_frame, mfi_new_cols)
show_fig(
    plot_price_with_features(
        mfi_frame,
        title=f'Close vs MFI ({MFI_WINDOW})',
        feature_cols=mfi_new_cols,
        normalize=False,
    )
)


In [ ]:
RSI_WINDOWS = [24,48]
RSI_METHOD = 'wilder'

rsi_frame, rsi_new_cols = apply_feature(
    raw_frame,
    add_rsi_features,
    price_col='close',
    windows=RSI_WINDOWS,
    method=RSI_METHOD,
)
show_feature_table(rsi_frame, rsi_new_cols)
show_fig(
    plot_price_with_features(
        rsi_frame,
        title=f'Close vs RSI {RSI_WINDOWS}',
        feature_cols=rsi_new_cols,
        normalize=False,
    )
)


In [ ]:
STOCH_WINDOW = 14
STOCH_SMOOTH = 3

stoch_frame, stoch_new_cols = apply_feature(
    raw_frame,
    add_stochastic_features,
    price_col='close',
    high_col='high',
    low_col='low',
    window=STOCH_WINDOW,
    smooth=STOCH_SMOOTH,
)
show_feature_table(stoch_frame, stoch_new_cols)
show_fig(
    plot_price_with_features(
        stoch_frame,
        title=f'Close vs Stochastic ({STOCH_WINDOW}, {STOCH_SMOOTH})',
        feature_cols=stoch_new_cols,
        normalize=False,
    )
)


In [ ]:
PRICE_MOM_WINDOWS = [5, 20, 60]

price_momentum_frame, price_momentum_new_cols = apply_feature(
    raw_frame,
    add_price_momentum_features,
    price_col='close',
    windows=PRICE_MOM_WINDOWS,
)
show_feature_table(price_momentum_frame, price_momentum_new_cols)
show_fig(
    plot_price_with_features(
        price_momentum_frame,
        title=f'Close vs Price Momentum {PRICE_MOM_WINDOWS}',
        feature_cols=price_momentum_new_cols,
        normalize=False,
    )
)


In [ ]:
RETURN_MOM_RETURNS_COL = 'close_logret'
RETURN_MOM_WINDOWS = [5, 20, 60]

return_mom_base = add_close_returns(raw_frame, log=True)
return_mom_base = add_close_returns(return_mom_base, log=False)
return_momentum_frame, return_momentum_new_cols = apply_feature(
    return_mom_base,
    add_return_momentum_features,
    returns_col=RETURN_MOM_RETURNS_COL,
    windows=RETURN_MOM_WINDOWS,
)
show_feature_table(return_momentum_frame, return_momentum_new_cols)
show_fig(
    plot_price_with_features(
        return_momentum_frame,
        title=f'Close vs Return Momentum from {RETURN_MOM_RETURNS_COL}',
        feature_cols=return_momentum_new_cols,
        normalize=False,
    )
)


In [ ]:
VOL_NORM_MOM_RETURNS_COL = 'close_logret'
VOL_NORM_MOM_VOL_WINDOW = 20
VOL_NORM_MOM_WINDOWS = [5, 20, 60]
VOL_NORM_MOM_EPS = 1e-8

vol_norm_base = add_close_returns(raw_frame, log=True)
vol_norm_base = add_close_returns(vol_norm_base, log=False)
vol_norm_base = add_volatility_features(
    vol_norm_base,
    returns_col=VOL_NORM_MOM_RETURNS_COL,
    rolling_windows=[VOL_NORM_MOM_VOL_WINDOW],
    ewma_spans=[],
    annualization_factor=None,
)
vol_norm_momentum_frame, vol_norm_momentum_new_cols = apply_feature(
    vol_norm_base,
    add_vol_normalized_momentum_features,
    returns_col=VOL_NORM_MOM_RETURNS_COL,
    vol_col=f'vol_rolling_{VOL_NORM_MOM_VOL_WINDOW}',
    windows=VOL_NORM_MOM_WINDOWS,
    eps=VOL_NORM_MOM_EPS,
)
show_feature_table(vol_norm_momentum_frame, vol_norm_momentum_new_cols)
show_fig(
    plot_price_with_features(
        vol_norm_momentum_frame,
        title='Close vs Vol-Normalized Momentum',
        feature_cols=vol_norm_momentum_new_cols,
        normalize=False,
    )
)


In [ ]:
SESSION_TIMEZONE = 'UTC'
SESSION_ADD_CYCLICAL_TIME = True
SESSION_INCLUDE_WEEKEND_FLAG = True
SESSION_CUSTOM_SESSIONS = None
SESSION_PLOT_COLUMNS = ['hour_sin_24', 'hour_cos_24', 'day_of_week_sin_7', 'day_of_week_cos_7', 'session_asia', 'session_europe', 'session_us', 'session_europe_us_overlap', 'is_weekend']

session_frame, session_new_cols = apply_feature(
    raw_frame,
    add_session_context_features,
    timezone=SESSION_TIMEZONE,
    add_cyclical_time=SESSION_ADD_CYCLICAL_TIME,
    include_weekend_flag=SESSION_INCLUDE_WEEKEND_FLAG,
    sessions=SESSION_CUSTOM_SESSIONS,
)
show_feature_table(session_frame, session_new_cols)
show_fig(
    plot_price_with_features(
        session_frame,
        title=f'Session context ({SESSION_TIMEZONE})',
        feature_cols=existing_columns(session_frame, SESSION_PLOT_COLUMNS),
        normalize=False,
    )
)


In [ ]:
REGIME_RETURNS_COL = 'close_ret'
REGIME_VOL_SHORT_WINDOW = 24
REGIME_VOL_LONG_WINDOW = 168
REGIME_TREND_FAST_SPAN = 24
REGIME_TREND_SLOW_SPAN = 72
REGIME_VOL_RATIO_HIGH_THRESHOLD = 1.25
REGIME_VOL_RATIO_LOW_THRESHOLD = 0.85

regime_base = add_close_returns(raw_frame, log=True)
regime_base = add_close_returns(regime_base, log=False)
regime_frame, regime_new_cols = apply_feature(
    regime_base,
    add_regime_context_features,
    price_col='close',
    returns_col=REGIME_RETURNS_COL,
    vol_short_window=REGIME_VOL_SHORT_WINDOW,
    vol_long_window=REGIME_VOL_LONG_WINDOW,
    trend_fast_span=REGIME_TREND_FAST_SPAN,
    trend_slow_span=REGIME_TREND_SLOW_SPAN,
    vol_ratio_high_threshold=REGIME_VOL_RATIO_HIGH_THRESHOLD,
    vol_ratio_low_threshold=REGIME_VOL_RATIO_LOW_THRESHOLD,
)
show_feature_table(regime_frame, regime_new_cols)
show_fig(
    plot_price_with_features(
        regime_frame,
        title='Regime context features',
        feature_cols=regime_new_cols,
        normalize=False,
    )
)


In [ ]:
SHOCK_RETURNS_COL = 'close_logret'
SHOCK_EMA_WINDOW = 24
SHOCK_ATR_WINDOW = 24
SHOCK_SHORT_HORIZON = 1
SHOCK_MEDIUM_HORIZON = 4
SHOCK_VOL_WINDOW = 24
SHOCK_RET_Z_THRESHOLD = 2.0
SHOCK_ATR_MULT_THRESHOLD = 1.5
SHOCK_DISTANCE_FROM_MEAN_THRESHOLD = 1.0
SHOCK_POST_ACTIVE_BARS = 1
SHOCK_USE_LOG_RETURNS = True
SHOCK_PLOT_COLUMNS = ['shock_ret_z_1h', 'shock_ret_z_4h', 'shock_atr_multiple_1h', 'shock_distance_ema', 'shock_active_window', 'shock_strength', 'bars_since_shock']

shock_base = add_close_returns(raw_frame, log=True)
shock_base = add_close_returns(shock_base, log=False)
shock_frame, shock_new_cols = apply_feature(
    shock_base,
    add_shock_context_features,
    price_col='close',
    high_col='high',
    low_col='low',
    returns_col=SHOCK_RETURNS_COL,
    ema_col=None,
    ema_window=SHOCK_EMA_WINDOW,
    atr_col=None,
    atr_window=SHOCK_ATR_WINDOW,
    short_horizon=SHOCK_SHORT_HORIZON,
    medium_horizon=SHOCK_MEDIUM_HORIZON,
    vol_window=SHOCK_VOL_WINDOW,
    ret_z_threshold=SHOCK_RET_Z_THRESHOLD,
    atr_mult_threshold=SHOCK_ATR_MULT_THRESHOLD,
    distance_from_mean_threshold=SHOCK_DISTANCE_FROM_MEAN_THRESHOLD,
    post_shock_active_bars=SHOCK_POST_ACTIVE_BARS,
    use_log_returns=SHOCK_USE_LOG_RETURNS,
)
show_feature_table(shock_frame, shock_new_cols)
show_fig(
    plot_price_with_features(
        shock_frame,
        title='Shock context features',
        feature_cols=existing_columns(shock_frame, SHOCK_PLOT_COLUMNS),
        normalize=False,
    )
)


In [ ]:
SR_WINDOWS = [24, 72]
SR_ATR_WINDOW = 24
SR_INCLUDE_PCT_DISTANCE = True
SR_INCLUDE_ATR_DISTANCE = True
SR_DISTANCE_PLOT_COLUMNS = ['support_distance_pct_24', 'resistance_distance_pct_24', 'support_distance_atr_24', 'resistance_distance_atr_24', 'support_distance_pct_72', 'resistance_distance_pct_72']

sr_frame, sr_new_cols = apply_feature(
    raw_frame,
    add_support_resistance_features,
    price_col='close',
    high_col='high',
    low_col='low',
    windows=SR_WINDOWS,
    atr_col=None,
    atr_window=SR_ATR_WINDOW,
    include_pct_distance=SR_INCLUDE_PCT_DISTANCE,
    include_atr_distance=SR_INCLUDE_ATR_DISTANCE,
)
sr_overlay_cols = ['close'] + [f'support_{window}' for window in SR_WINDOWS] + [f'resistance_{window}' for window in SR_WINDOWS]
show_feature_table(sr_frame, sr_new_cols)
show_fig(plot_price_overlay(sr_frame, existing_columns(sr_frame, sr_overlay_cols), title=f'Close with support / resistance {SR_WINDOWS}'))
show_fig(
    plot_price_with_features(
        sr_frame,
        title='Support / resistance distances',
        feature_cols=existing_columns(sr_frame, SR_DISTANCE_PLOT_COLUMNS),
        normalize=False,
    )
)


In [ ]:
SR2_ATR_WINDOW = 24
SR2_PIVOT_LEFT_WINDOW = 24
SR2_PIVOT_CONFIRM_BARS = 6
SR2_TOUCH_TOLERANCE_ATR = 0.25
SR2_BREAKOUT_TOLERANCE_ATR = 0.05
SR2_PLOT_COLUMNS = ['sr_v2_support_age', 'sr_v2_resistance_age', 'sr_v2_support_touch_count', 'sr_v2_resistance_touch_count', 'sr_v2_breakout_up', 'sr_v2_breakout_down', 'sr_v2_support_distance_atr', 'sr_v2_resistance_distance_atr']

sr2_frame, sr2_new_cols = apply_feature(
    raw_frame,
    add_support_resistance_v2_features,
    price_col='close',
    high_col='high',
    low_col='low',
    atr_col=None,
    atr_window=SR2_ATR_WINDOW,
    pivot_left_window=SR2_PIVOT_LEFT_WINDOW,
    pivot_confirm_bars=SR2_PIVOT_CONFIRM_BARS,
    touch_tolerance_atr=SR2_TOUCH_TOLERANCE_ATR,
    breakout_tolerance_atr=SR2_BREAKOUT_TOLERANCE_ATR,
)
sr2_overlay_cols = ['close', 'sr_v2_support_level', 'sr_v2_resistance_level']
show_feature_table(sr2_frame, sr2_new_cols)
show_fig(plot_price_overlay(sr2_frame, existing_columns(sr2_frame, sr2_overlay_cols), title='Close with pivot-based support / resistance'))
show_fig(
    plot_price_with_features(
        sr2_frame,
        title='Support / resistance v2 features',
        feature_cols=existing_columns(sr2_frame, SR2_PLOT_COLUMNS),
        normalize=False,
    )
)


In [ ]:
MACRO_COLUMNS = ['close', 'volume']
MACRO_AVAILABILITY_LAG = 1
MACRO_LAGS = [1, 24]
MACRO_PCT_CHANGE_PERIODS = [1, 24]
MACRO_ZSCORE_WINDOW = 48
MACRO_EMA_SPANS = [12]
MACRO_ALLOW_MISSING = False
MACRO_PLOT_COLUMNS = ['close_avail_lag_1', 'close_pct_24', 'close_z_48', 'close_ema_gap_12', 'volume_pct_24', 'volume_z_48', 'volume_ema_gap_12']

macro_frame, macro_new_cols = apply_feature(
    raw_frame,
    add_macro_context_features,
    columns=MACRO_COLUMNS,
    availability_lag=MACRO_AVAILABILITY_LAG,
    lags=MACRO_LAGS,
    pct_change_periods=MACRO_PCT_CHANGE_PERIODS,
    zscore_window=MACRO_ZSCORE_WINDOW,
    ema_spans=MACRO_EMA_SPANS,
    allow_missing=MACRO_ALLOW_MISSING,
)
show_feature_table(macro_frame, macro_new_cols)
show_fig(
    plot_price_with_features(
        macro_frame,
        title='Macro / exogenous context features',
        feature_cols=existing_columns(macro_frame, MACRO_PLOT_COLUMNS),
        normalize=False,
    )
)


In [ ]:
FEATURE_TRANSFORMS = [
    {'kind': 'rolling_zscore', 'source_col': 'close', 'window': 48, 'shift': 1, 'ddof': 0, 'output_col': 'close_z_48'},
    {'kind': 'rolling_clip', 'source_col': 'volume', 'window': 48, 'lower_q': 0.05, 'upper_q': 0.95, 'shift': 1, 'output_col': 'volume_clip_48'},
    {'kind': 'ratio', 'numerator_col': 'close', 'denominator_col': 'open', 'eps': 1e-8, 'output_col': 'close_open_ratio'},
]

transform_frame, transform_new_cols = apply_feature(
    raw_frame,
    add_feature_transforms,
    transforms=FEATURE_TRANSFORMS,
)
show_feature_table(transform_frame, transform_new_cols)
show_fig(
    plot_price_with_features(
        transform_frame,
        title='Feature transforms',
        feature_cols=transform_new_cols,
        normalize=False,
    )
)


In [ ]:
combo_work = raw_frame.copy()
combo_work, _ = apply_feature(combo_work, add_trend_features, price_col='close', sma_windows=[20, 50], ema_spans=[9, 21])
combo_work, _ = apply_feature(combo_work, add_bollinger_features, price_col='close', window=20, n_std=2.0)
combo_work, _ = apply_feature(combo_work, add_rsi_features, price_col='close', windows=[9, 14], method='wilder')
combo_work, _ = apply_feature(combo_work, add_macd_features, price_col='close', fast=12, slow=26, signal=9)

COMBO_FEATURE_COLUMNS = ['close_rsi_9', 'bb_percent_b_20_2.0', 'macd_12_26', 'macd_hist_12_26_9']
COMBO_WEIGHTS = {
    'close_rsi_9': 1.0,
    'bb_percent_b_20_2.0': 1.0,
    'macd_12_26': 1.0,
    'macd_hist_12_26_9': 0.8,
}

show_feature_table(combo_work, existing_columns(combo_work, COMBO_FEATURE_COLUMNS))
show_fig(
    plot_price_with_feature_combo(
        combo_work,
        title='Close vs Feature Combo',
        feature_cols=existing_columns(combo_work, COMBO_FEATURE_COLUMNS),
        feature_weights=COMBO_WEIGHTS,
        normalize=True,
    )
)
